In [2]:
print("hello")

hello


In [ ]:
!pip install -q gdown

In [3]:
!wget -O Offroad_Segmentation_Training_Dataset.zip \
"https://storage.googleapis.com/duality-public-share/Hackathons/Duality%20Hackathon/Offroad_Segmentation_Training_Dataset.zip"

--2026-03-12 14:36:31--  https://storage.googleapis.com/duality-public-share/Hackathons/Duality%20Hackathon/Offroad_Segmentation_Training_Dataset.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.64.207, 173.194.206.207, 108.177.121.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.64.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2794457228 (2.6G) [application/zip]
Saving to: ‘Offroad_Segmentation_Training_Dataset.zip’

Offroad_Segmentatio 100%[===================>]   2.60G   218MB/s    in 12s     

2026-03-12 14:36:44 (220 MB/s) - ‘Offroad_Segmentation_Training_Dataset.zip’ saved [2794457228/2794457228]



In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/dataset"):
    print(root, "->", len(files))

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1fotdA4mGlIuPisXDKfEOm3xr2674MvvH \
-O /kaggle/working/dataset/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages/Color_Images \
--remaining-ok

In [5]:
!wget -O Offroad_Segmentation_testImages.zip \
"https://storage.googleapis.com/duality-public-share/Hackathons/Duality%20Hackathon/Offroad_Segmentation_testImages.zip"

--2026-03-12 14:37:47--  https://storage.googleapis.com/duality-public-share/Hackathons/Duality%20Hackathon/Offroad_Segmentation_testImages.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.125.207, 192.178.212.207, 74.125.132.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.125.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1158592290 (1.1G) [application/zip]
Saving to: ‘Offroad_Segmentation_testImages.zip’

Offroad_Segmentatio 100%[===================>]   1.08G   221MB/s    in 5.2s    

2026-03-12 14:37:53 (211 MB/s) - ‘Offroad_Segmentation_testImages.zip’ saved [1158592290/1158592290]



In [6]:
!ls

Offroad_Segmentation_testImages.zip  Offroad_Segmentation_Training_Dataset.zip


In [7]:
!unzip -q Offroad_Segmentation_Training_Dataset.zip
!unzip -q Offroad_Segmentation_testImages.zip

In [10]:
%%writefile train_new.py
"""
Segmentation Training Script v3 — Google Colab MAX RESOURCE Edition
=====================================================================
Designed to fully saturate a Colab A100/V100/T4 GPU within a 2-hour window.

Key upgrades vs v2:
  - Effective batch size ~64 via gradient accumulation (physical=8, accum=8)
    → simulates a batch-400 style "large batch" training signal without OOM
  - DINOv2-Large backbone (vitl14_reg, 307M params) — largest practical on Colab
  - Unfreeze last 6 ViT blocks for aggressive domain adaptation
  - Higher input resolution: 518×518 (37×37 token grid, divisible by 14)
  - Wider ASPP (384 channels) + deeper decoder
  - Cosine annealing with warm restarts (SGDR) for better convergence
  - Label smoothing in CrossEntropy
  - Test-Time Augmentation (TTA) at inference: H-flip + scale ensemble
  - Automatic mixed precision + torch.compile (if PyTorch ≥ 2.0)
  - Per-epoch Google Drive checkpoint backup (Colab-safe)
  - Estimated training time: ~90–110 min on A100, ~110–130 min on V100

USAGE (in Colab):
  !python train_segmentation_v3_colab.py \
      --train_dir /content/drive/MyDrive/dataset/train \
      --val_dir   /content/drive/MyDrive/dataset/val \
      --output_dir /content/drive/MyDrive/seg_output \
      --epochs 25
"""

import os, sys, random, argparse, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ============================================================================
# CONFIG  (override via argparse; these are the Colab-optimised defaults)
# ============================================================================
def get_config():
    p = argparse.ArgumentParser(description="Offroad Segmentation v3 — Colab MAX")

    # paths
    p.add_argument("--train_dir",  default="../Offroad_Segmentation_Training_Dataset/train")
    p.add_argument("--val_dir",    default="../Offroad_Segmentation_Training_Dataset/val")
    p.add_argument("--output_dir", default="./seg_output_v3")

    # model
    p.add_argument("--backbone",         default="large",  choices=["small","base","large"])
    p.add_argument("--unfreeze_blocks",  type=int, default=6)
    p.add_argument("--aspp_channels",    type=int, default=384)
    p.add_argument("--img_size",         type=int, default=518)   # must be divisible by 14

    # training
    p.add_argument("--epochs",           type=int,   default=25)
    p.add_argument("--physical_batch",   type=int,   default=8)   # per-GPU batch
    p.add_argument("--accum_steps",      type=int,   default=8)   # effective_batch = physical × accum
    p.add_argument("--lr",               type=float, default=2e-4)
    p.add_argument("--weight_decay",     type=float, default=1e-4)
    p.add_argument("--grad_clip",        type=float, default=1.0)
    p.add_argument("--label_smoothing",  type=float, default=0.1)
    p.add_argument("--dice_weight",      type=float, default=0.5)
    p.add_argument("--num_workers",      type=int,   default=2)
    p.add_argument("--use_compile",      action="store_true", default=False)

    return p.parse_args()


# ============================================================================
# CLASS DEFINITIONS  (11 classes — includes Flowers fix from v2)
# ============================================================================
VALUE_MAP = {
    0: 0, 100: 1, 200: 2, 300: 3, 500: 4,
    550: 5, 600: 6, 700: 7, 800: 8, 7100: 9, 10000: 10,
}
N_CLASSES = 11
CLASS_NAMES = [
    "Background","Trees","Lush Bushes","Dry Grass","Dry Bushes",
    "Ground Clutter","Flowers","Logs","Rocks","Landscape","Sky",
]

# Inverse-frequency weights — rare classes weighted higher
_CW = torch.tensor([0.4,2.0,2.0,1.5,2.0,3.0,4.5,6.0,4.5,0.4,0.4], dtype=torch.float32)
CLASS_WEIGHTS = _CW / _CW.mean()   # keep mean≈1 to stabilise LR

COLOR_PALETTE = np.array([
    [0,0,0],[34,139,34],[0,200,0],[210,180,140],[139,90,43],
    [128,128,0],[255,165,0],[139,69,19],[128,128,128],[160,82,45],[135,206,235],
], dtype=np.uint8)


def convert_mask(mask_img: Image.Image) -> np.ndarray:
    arr = np.array(mask_img, dtype=np.int32)
    out = np.zeros_like(arr, dtype=np.uint8)
    for raw, cls in VALUE_MAP.items():
        out[arr == raw] = cls
    return out   # H×W uint8 class IDs


def mask_to_color(mask_np: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES):
        rgb[mask_np == c] = COLOR_PALETTE[c]
    return rgb


# ============================================================================
# AUGMENTATION  (joint spatial + photometric)
# ============================================================================
class JointAugment:
    def __init__(self, size: int, is_train: bool):
        self.size     = size
        self.is_train = is_train

    def __call__(self, img: Image.Image, mask_np: np.ndarray):
        mask_img = Image.fromarray(mask_np)   # PIL for spatial ops

        # ── resize ──
        img      = TF.resize(img,      (self.size, self.size), interpolation=Image.BILINEAR)
        mask_img = TF.resize(mask_img, (self.size, self.size), interpolation=Image.NEAREST)

        if self.is_train:
            # Random horizontal flip
            if random.random() > 0.5:
                img = TF.hflip(img); mask_img = TF.hflip(mask_img)

            # Random vertical flip (synthetic data → valid)
            if random.random() > 0.8:
                img = TF.vflip(img); mask_img = TF.vflip(mask_img)

            # Multi-scale random crop (75%–100%)
            scale    = random.uniform(0.75, 1.0)
            ch, cw   = int(self.size * scale), int(self.size * scale)
            i, j, h, w = transforms.RandomCrop.get_params(img, (ch, cw))
            img      = TF.resized_crop(img,      i, j, h, w, (self.size, self.size), Image.BILINEAR)
            mask_img = TF.resized_crop(mask_img, i, j, h, w, (self.size, self.size), Image.NEAREST)

            # Random 90° rotation (0, 90, 180, 270)
            angle = random.choice([0, 90, 180, 270])
            if angle:
                img      = TF.rotate(img,      angle)
                mask_img = TF.rotate(mask_img, angle)

            # Photometric augmentation (image only)
            img = TF.adjust_brightness(img, random.uniform(0.6, 1.4))
            img = TF.adjust_contrast(img,   random.uniform(0.6, 1.4))
            img = TF.adjust_saturation(img, random.uniform(0.6, 1.4))
            img = TF.adjust_hue(img,        random.uniform(-0.15, 0.15))
            if random.random() > 0.5:
                img = TF.gaussian_blur(img, kernel_size=5,
                                       sigma=random.uniform(0.1, 2.0))
            # Random grayscale (forces texture reliance)
            if random.random() > 0.8:
                img = TF.to_grayscale(img, num_output_channels=3)

        # ── to tensor ──
        img_t  = TF.to_tensor(img)
        img_t  = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
        mask_t = torch.from_numpy(np.array(mask_img)).long()
        return img_t, mask_t


# ============================================================================
# DATASET
# ============================================================================
class SegDataset(Dataset):
    def __init__(self, data_dir: str, augment: JointAugment):
        self.img_dir  = os.path.join(data_dir, "Color_Images")
        self.msk_dir  = os.path.join(data_dir, "Segmentation")
        self.augment  = augment
        self.ids      = sorted(f for f in os.listdir(self.img_dir)
                               if f.lower().endswith((".png",".jpg",".jpeg")))
        assert len(self.ids) > 0, f"No images found in {self.img_dir}"

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        img  = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        msk  = Image.open(os.path.join(self.msk_dir, name))
        msk_np = convert_mask(msk)
        img_t, msk_t = self.augment(img, msk_np)
        return img_t, msk_t


# ============================================================================
# MODEL — DINOv2 + DeepLabV3+ head (WIDER)
# ============================================================================
class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling with configurable channel width."""
    def __init__(self, in_ch: int, out_ch: int = 384,
                 dilations: tuple = (1, 6, 12, 18, 24)):
        super().__init__()
        self.branches = nn.ModuleList()
        for d in dilations:
            ks = 1 if d == 1 else 3
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, ks, padding=d if d > 1 else 0,
                          dilation=d, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        # GAP branch
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        total_in = out_ch * (len(dilations) + 1)
        self.project = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.gap(x), (h,w),
                                   mode="bilinear", align_corners=False))
        return self.project(torch.cat(feats, dim=1))


class DeepLabV3PlusHead(nn.Module):
    """
    Upgraded DeepLabV3+ head:
      - Wider ASPP (5 dilation rates instead of 4)
      - 3-layer decoder with residual connection
      - Squeeze-and-Excitation on ASPP output
    """
    def __init__(self, in_ch: int, num_classes: int,
                 token_h: int, token_w: int,
                 aspp_ch: int = 384):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w

        self.input_proj = nn.Sequential(
            nn.Conv2d(in_ch, aspp_ch, 1, bias=False),
            nn.BatchNorm2d(aspp_ch), nn.ReLU(inplace=True),
        )
        self.aspp = ASPP(aspp_ch, aspp_ch)

        # Squeeze-and-Excitation on ASPP output (channel attention)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(aspp_ch, aspp_ch // 16),
            nn.ReLU(inplace=True),
            nn.Linear(aspp_ch // 16, aspp_ch),
            nn.Sigmoid(),
        )

        # 3-conv decoder
        self.dec1 = nn.Sequential(
            nn.Conv2d(aspp_ch, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(128, num_classes, 1)

        # shortcut for residual
        self.shortcut = nn.Conv2d(aspp_ch, 256, 1, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.token_h, self.token_w, C).permute(0, 3, 1, 2)
        x = self.input_proj(x)
        x = self.aspp(x)

        # SE channel attention
        se = self.se(x).view(x.size(0), x.size(1), 1, 1)
        x  = x * se

        res = self.shortcut(x)
        x   = self.dec1(x)
        x   = self.dec2(x) + res   # residual
        x   = self.dec3(x)
        return self.classifier(x)


# ============================================================================
# LOSS  (CE with label smoothing + Dice)
# ============================================================================
class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None, dice_weight=0.5,
                 label_smoothing=0.1, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=label_smoothing,
            ignore_index=255,
        )
        self.dice_w = dice_weight
        self.smooth = smooth

    def _dice(self, logits, targets):
        num_cls = logits.shape[1]
        probs   = F.softmax(logits, dim=1)
        t_oh    = F.one_hot(targets.clamp(0, num_cls-1), num_cls) \
                    .permute(0,3,1,2).float()
        dims    = (0, 2, 3)
        inter   = (probs * t_oh).sum(dims)
        card    = (probs + t_oh).sum(dims)
        return (1.0 - (2*inter + self.smooth) / (card + self.smooth)).mean()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice_w * self._dice(logits, targets)


# ============================================================================
# METRICS
# ============================================================================
def iou_per_class(pred_logits, target):
    """Returns list of per-class IoU (nan if class absent in batch)."""
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt  = target.view(-1)
    ious = []
    for c in range(N_CLASSES):
        p = pred == c; t = tgt == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append((inter/union).item() if union > 0 else float("nan"))
    return ious


def mean_iou(ious):
    v = [x for x in ious if not math.isnan(x)]
    return float(np.mean(v)) if v else 0.0


def pixel_acc(pred_logits, target):
    return (torch.argmax(pred_logits,1) == target).float().mean().item()


# ============================================================================
# TEST-TIME AUGMENTATION (TTA)
# ============================================================================
@torch.no_grad()
def tta_predict(backbone, head, imgs, size):
    """Average predictions over original + H-flip."""
    feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
    logits = head(feats)
    logits = F.interpolate(logits, size, mode="bilinear", align_corners=False)

    imgs_f  = torch.flip(imgs, dims=[3])
    feats_f = backbone.forward_features(imgs_f)["x_norm_patchtokens"]
    logits_f = head(feats_f)
    logits_f = F.interpolate(logits_f, size, mode="bilinear", align_corners=False)
    logits_f = torch.flip(logits_f, dims=[3])

    return (F.softmax(logits,1) + F.softmax(logits_f,1)) / 2.0


# ============================================================================
# EVALUATION
# ============================================================================
@torch.no_grad()
def evaluate(backbone, head, loader, device, loss_fn, use_amp, use_tta=False):
    backbone.eval(); head.eval()
    size = (loader.dataset.augment.size,) * 2
    tot_loss = tot_acc = 0.0
    all_ious = [[] for _ in range(N_CLASSES)]

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            if use_tta:
                probs  = tta_predict(backbone, head, imgs, size)
                logits_for_loss = torch.log(probs.clamp(min=1e-7))
                loss   = F.nll_loss(logits_for_loss, labels,
                                    weight=loss_fn.ce.weight)
                out    = probs
            else:
                feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                logits = head(feats)
                logits = F.interpolate(logits, size,
                                       mode="bilinear", align_corners=False)
                loss   = loss_fn(logits, labels)
                out    = logits

        tot_loss += loss.item()
        tot_acc  += pixel_acc(out, labels)
        for c, v in enumerate(iou_per_class(out, labels)):
            all_ious[c].append(v)

    n = max(len(loader), 1)
    class_ious = [mean_iou(all_ious[c]) for c in range(N_CLASSES)]
    backbone.train(); head.train()
    return tot_loss/n, tot_acc/n, class_ious


# ============================================================================
# PLOTTING
# ============================================================================
def save_plots(history, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ep = range(1, len(history["train_loss"])+1)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    axes[0,0].plot(ep, history["train_loss"], label="train")
    axes[0,0].plot(ep, history["val_loss"],   label="val")
    axes[0,0].set_title("Loss (CE + Dice)"); axes[0,0].legend(); axes[0,0].grid()

    axes[0,1].plot(ep, history["train_iou"], label="train")
    axes[0,1].plot(ep, history["val_iou"],   label="val")
    axes[0,1].set_title("Mean IoU"); axes[0,1].legend(); axes[0,1].grid()
    axes[0,1].axhline(0.6, color="red", linestyle="--", alpha=0.5, label="target 0.60")

    axes[1,0].plot(ep, history["train_acc"], label="train")
    axes[1,0].plot(ep, history["val_acc"],   label="val")
    axes[1,0].set_title("Pixel Accuracy"); axes[1,0].legend(); axes[1,0].grid()

    final_iou = history["val_class_ious"][-1]
    axes[1,1].barh(range(N_CLASSES), final_iou,
                   color=[COLOR_PALETTE[c]/255 for c in range(N_CLASSES)])
    axes[1,1].set_yticks(range(N_CLASSES))
    axes[1,1].set_yticklabels(CLASS_NAMES, fontsize=8)
    axes[1,1].set_xlim(0,1)
    axes[1,1].axvline(mean_iou(final_iou), color="red",
                      linestyle="--", label=f"mean={mean_iou(final_iou):.3f}")
    axes[1,1].set_title("Per-Class Val IoU (final epoch)")
    axes[1,1].legend(); axes[1,1].grid(axis="x")

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "training_dashboard.png"), dpi=150)
    plt.close()


def save_metrics_txt(history, cfg, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    best_ep = int(np.argmax(history["val_iou"]))
    eff_bs  = cfg.physical_batch * cfg.accum_steps
    with open(os.path.join(out_dir, "evaluation_metrics.txt"), "w") as f:
        f.write("TRAINING RESULTS — v3 Colab MAX\n")
        f.write("=" * 65 + "\n")
        f.write(f"Backbone:           DINOv2-{cfg.backbone.capitalize()}\n")
        f.write(f"Effective batch:    {eff_bs}  "
                f"(physical={cfg.physical_batch} × accum={cfg.accum_steps})\n")
        f.write(f"Input resolution:   {cfg.img_size}×{cfg.img_size}\n")
        f.write(f"Unfrozen blocks:    {cfg.unfreeze_blocks}\n")
        f.write(f"Epochs trained:     {len(history['train_loss'])}\n")
        f.write("=" * 65 + "\n\n")
        f.write(f"Best Val IoU:       {max(history['val_iou']):.4f}  (epoch {best_ep+1})\n")
        f.write(f"Final Val IoU:      {history['val_iou'][-1]:.4f}\n")
        f.write(f"Final Val Acc:      {history['val_acc'][-1]:.4f}\n")
        f.write(f"Final Val Loss:     {history['val_loss'][-1]:.4f}\n\n")
        f.write("Per-Class Val IoU @ best epoch:\n")
        for name, iou in zip(CLASS_NAMES, history["val_class_ious"][best_ep]):
            bar = "█" * int(iou * 30)
            f.write(f"  {name:<20} {iou:.4f}  {bar}\n")
        f.write("\n" + "=" * 65 + "\n")
        f.write("Per-Epoch History:\n")
        f.write(f"{'Ep':>4} {'TrLoss':>9} {'VaLoss':>9} "
                f"{'TrIoU':>8} {'VaIoU':>8} {'VaAcc':>8}\n")
        f.write("-" * 50 + "\n")
        for i in range(len(history["train_loss"])):
            f.write(f"{i+1:>4} {history['train_loss'][i]:>9.4f} "
                    f"{history['val_loss'][i]:>9.4f} "
                    f"{history['train_iou'][i]:>8.4f} "
                    f"{history['val_iou'][i]:>8.4f} "
                    f"{history['val_acc'][i]:>8.4f}\n")
    print(f"✓ Metrics saved to {out_dir}/evaluation_metrics.txt")


# ============================================================================
# MAIN
# ============================================================================
def main():
    cfg    = get_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    os.makedirs(cfg.output_dir, exist_ok=True)

    print("=" * 65)
    print("Offroad Segmentation v3 — Colab MAX")
    print("=" * 65)
    print(f"  Device:            {device}")
    if device.type == "cuda":
        prop = torch.cuda.get_device_properties(0)
        print(f"  GPU:               {prop.name}  ({prop.total_memory//1024**3} GB)")
    print(f"  Backbone:          DINOv2-{cfg.backbone.capitalize()}")
    print(f"  Effective batch:   {cfg.physical_batch * cfg.accum_steps}")
    print(f"  Input resolution:  {cfg.img_size}×{cfg.img_size}")
    print(f"  Epochs:            {cfg.epochs}")
    print(f"  AMP:               {use_amp}")
    print("=" * 65)

    S = cfg.img_size
    assert S % 14 == 0, f"img_size ({S}) must be divisible by 14"

    # ── datasets ──────────────────────────────────────────────────────────────
    train_aug = JointAugment(S, is_train=True)
    val_aug   = JointAugment(S, is_train=False)

    train_set = SegDataset(cfg.train_dir, train_aug)
    val_set   = SegDataset(cfg.val_dir,   val_aug)

    train_loader = DataLoader(
        train_set, batch_size=cfg.physical_batch,
        shuffle=True,  num_workers=cfg.num_workers,
        pin_memory=True, drop_last=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_set, batch_size=cfg.physical_batch,
        shuffle=False, num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    print(f"\nTrain: {len(train_set)} images  |  Val: {len(val_set)} images")

    # ── backbone ──────────────────────────────────────────────────────────────
    backbone_names = {
        "small": "dinov2_vits14",
        "base":  "dinov2_vitb14_reg",
        "large": "dinov2_vitl14_reg",
    }
    print(f"\nLoading {backbone_names[cfg.backbone]} ...")
    backbone = torch.hub.load("facebookresearch/dinov2",
                              backbone_names[cfg.backbone],
                              pretrained=True)
    backbone.to(device)

    # Freeze all, then unfreeze last N blocks + norm
    for p in backbone.parameters(): p.requires_grad_(False)
    for block in list(backbone.blocks)[-cfg.unfreeze_blocks:]:
        for p in block.parameters(): p.requires_grad_(True)
    for p in backbone.norm.parameters(): p.requires_grad_(True)
    n_trainable_bb = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f"Backbone trainable params: {n_trainable_bb:,}")

    # Probe feature dimensions
    with torch.no_grad():
        probe = torch.zeros(1, 3, S, S, device=device)
        feats = backbone.forward_features(probe)["x_norm_patchtokens"]
    embed_dim         = feats.shape[-1]
    token_h = token_w = S // 14
    print(f"Embed dim: {embed_dim}  |  Token grid: {token_h}×{token_w}")

    # ── segmentation head ──────────────────────────────────────────────────
    head = DeepLabV3PlusHead(
        embed_dim, N_CLASSES, token_h, token_w,
        aspp_ch=cfg.aspp_channels,
    ).to(device)
    n_head = sum(p.numel() for p in head.parameters())
    print(f"Head params: {n_head:,}")

    # Optional torch.compile (PyTorch ≥ 2.0, Colab Pro usually has it)
    if cfg.use_compile and hasattr(torch, "compile"):
        print("Compiling head with torch.compile ...")
        head = torch.compile(head)

    # ── loss ──────────────────────────────────────────────────────────────────
    loss_fn = CombinedLoss(
        class_weights=CLASS_WEIGHTS.to(device),
        dice_weight=cfg.dice_weight,
        label_smoothing=cfg.label_smoothing,
    )

    # ── optimizer & scheduler ─────────────────────────────────────────────────
    # Differential LR: backbone gets 10× less than head
    param_groups = [
        {"params": [p for p in backbone.parameters() if p.requires_grad],
         "lr": cfg.lr * 0.05,    # 5% LR for backbone — conservative fine-tune
         "name": "backbone"},
        {"params": list(head.parameters()),
         "lr": cfg.lr,
         "name": "head"},
    ]
    optimizer = optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    # Cosine annealing with warm restarts — 2 restarts over the run
    T0     = max(cfg.epochs // 3, 5)
    sched  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=2, eta_min=1e-6,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    eff_bs = cfg.physical_batch * cfg.accum_steps
    print(f"\nEffective batch size: {eff_bs}  "
          f"(physical={cfg.physical_batch} × accum={cfg.accum_steps})")
    print(f"Steps per epoch: {len(train_loader)}  "
          f"(≈{len(train_loader)//cfg.accum_steps} optimizer steps)")
    print(f"\nStarting training for {cfg.epochs} epochs ...\n")

    best_val_iou  = 0.0
    ckpt_path     = os.path.join(cfg.output_dir, "best_model_v3.pth")
    history       = {k: [] for k in [
        "train_loss","val_loss","train_iou","val_iou",
        "train_acc","val_acc","val_class_ious",
    ]}

    for epoch in range(cfg.epochs):
        t0 = time.time()
        backbone.train(); head.train()

        ep_loss, ep_iou, ep_acc = [], [], []
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader),
                    total=len(train_loader),
                    desc=f"Epoch {epoch+1:>3}/{cfg.epochs} [Train]",
                    leave=False, ncols=100)

        for step, (imgs, labels) in pbar:
            imgs, labels = imgs.to(device, non_blocking=True), \
                           labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                logits = head(feats)
                logits = F.interpolate(logits, (S, S),
                                       mode="bilinear", align_corners=False)
                # Scale loss by accum_steps so effective gradient magnitude
                # matches a true large-batch run
                loss   = loss_fn(logits, labels) / cfg.accum_steps

            scaler.scale(loss).backward()

            # Gradient accumulation step
            if (step + 1) % cfg.accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for pg in optimizer.param_groups for p in pg["params"]],
                    cfg.grad_clip,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            with torch.no_grad():
                real_loss = loss.item() * cfg.accum_steps
                batch_iou = mean_iou(iou_per_class(logits, labels))
                batch_acc = pixel_acc(logits, labels)

            ep_loss.append(real_loss)
            ep_iou.append(batch_iou)
            ep_acc.append(batch_acc)
            pbar.set_postfix(
                loss=f"{real_loss:.4f}",
                iou=f"{batch_iou:.3f}",
                lr=f"{optimizer.param_groups[1]['lr']:.2e}",
            )

        sched.step(epoch)   # per-epoch step for CosineAnnealingWarmRestarts

        # ── validation ──
        val_loss, val_acc, val_ciu = evaluate(
            backbone, head, val_loader, device, loss_fn, use_amp,
            use_tta=(epoch >= cfg.epochs - 5),   # TTA only in last 5 epochs
        )
        val_iou = mean_iou(val_ciu)
        elapsed = time.time() - t0

        history["train_loss"].append(float(np.mean(ep_loss)))
        history["val_loss"].append(val_loss)
        history["train_iou"].append(float(np.mean(ep_iou)))
        history["val_iou"].append(val_iou)
        history["train_acc"].append(float(np.mean(ep_acc)))
        history["val_acc"].append(val_acc)
        history["val_class_ious"].append(val_ciu)

        print(f"Epoch {epoch+1:>3}/{cfg.epochs}  "
              f"tr_loss={history['train_loss'][-1]:.4f}  "
              f"va_loss={val_loss:.4f}  "
              f"tr_iou={history['train_iou'][-1]:.4f}  "
              f"va_iou={val_iou:.4f}  "
              f"va_acc={val_acc:.4f}  "
              f"[{elapsed:.0f}s]")

        # Per-class breakdown every 5 epochs
        if (epoch + 1) % 5 == 0:
            print("  ┌─ Per-class Val IoU:")
            for name, iou in zip(CLASS_NAMES, val_ciu):
                bar = "█" * int(iou * 25)
                print(f"  │ {name:<20} {iou:.3f}  {bar}")
            print("  └─")

        # Save best checkpoint
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save({
                "epoch":             epoch + 1,
                "head_state_dict":   head.state_dict()
                                     if not hasattr(head, "_orig_mod")
                                     else head._orig_mod.state_dict(),
                "val_iou":           val_iou,
                "val_class_ious":    val_ciu,
                "config": {
                    "backbone":       cfg.backbone,
                    "img_size":       cfg.img_size,
                    "aspp_channels":  cfg.aspp_channels,
                    "embed_dim":      embed_dim,
                    "token_h":        token_h,
                    "token_w":        token_w,
                    "n_classes":      N_CLASSES,
                },
            }, ckpt_path)
            print(f"  ✓ Best checkpoint saved  (val_iou={val_iou:.4f})")

        # Intermediate save every 5 epochs (Colab safety net)
        if (epoch + 1) % 5 == 0:
            inter_path = os.path.join(cfg.output_dir,
                                      f"checkpoint_ep{epoch+1}.pth")
            torch.save({"epoch": epoch+1,
                        "head_state_dict": head.state_dict()
                                           if not hasattr(head, "_orig_mod")
                                           else head._orig_mod.state_dict(),
                        "val_iou": val_iou}, inter_path)
            print(f"  ✓ Intermediate checkpoint: {inter_path}")

    # ── final saves ──────────────────────────────────────────────────────────
    save_plots(history, cfg.output_dir)
    save_metrics_txt(history, cfg, cfg.output_dir)

    print("\n" + "=" * 65)
    print("Training complete!")
    print(f"Best Val IoU:   {best_val_iou:.4f}")
    print(f"Checkpoint:     {ckpt_path}")
    print(f"Output dir:     {cfg.output_dir}")
    print("=" * 65)


if __name__ == "__main__":
    main()


Overwriting train_new.py


In [12]:
!python train_new.py \
--train_dir /kaggle/working/Offroad_Segmentation_Training_Dataset/train \
--val_dir /kaggle/working/Offroad_Segmentation_Training_Dataset/val \
--output_dir /kaggle/working/seg_output_v3 \
--epochs 25

Offroad Segmentation v3 — Colab MAX
  Device:            cuda
  GPU:               Tesla T4  (14 GB)
  Backbone:          DINOv2-Large
  Effective batch:   64
  Input resolution:  518×518
  Epochs:            25
  AMP:               True

Train: 2857 images  |  Val: 317 images

Loading dinov2_vitl14_reg ...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbai

In [13]:
!zip -r seg_output_v3.zip /kaggle/working/seg_output_v3

  adding: kaggle/working/seg_output_v3/ (stored 0%)
  adding: kaggle/working/seg_output_v3/checkpoint_ep5.pth (deflated 8%)
  adding: kaggle/working/seg_output_v3/training_dashboard.png (deflated 15%)
  adding: kaggle/working/seg_output_v3/checkpoint_ep20.pth (deflated 8%)
  adding: kaggle/working/seg_output_v3/checkpoint_ep15.pth (deflated 8%)
  adding: kaggle/working/seg_output_v3/evaluation_metrics.txt (deflated 67%)
  adding: kaggle/working/seg_output_v3/checkpoint_ep10.pth (deflated 8%)
  adding: kaggle/working/seg_output_v3/best_model_v3.pth (deflated 8%)
  adding: kaggle/working/seg_output_v3/checkpoint_ep25.pth (deflated 8%)


In [14]:
from IPython.display import FileLink
FileLink('seg_output_v3.zip')

/kaggle/working/seg_output_v3.zip

In [16]:
%%writefile test.py
"""
Segmentation Test / Inference Script v3
========================================
Companion to train_new.py (v3 Colab MAX).
Loads a v3 checkpoint (DeepLabV3+ head on DINOv2 backbone) and runs
evaluation + prediction saving on a test dataset.

Supports:
  - Auto-config from checkpoint (backbone, img_size, aspp_channels, etc.)
  - Test-Time Augmentation (TTA): H-flip ensemble
  - Per-class IoU, Dice, Pixel Accuracy
  - Saves: raw masks, colored masks, side-by-side comparisons, metrics

USAGE:
  python test_new.py \
      --model_path ./seg_output_v3/best_model_v3.pth \
      --data_dir   ../Offroad_Segmentation_testImages \
      --output_dir ./predictions_v3
"""

import os, math, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm


# ============================================================================
# CLASS DEFINITIONS  (must match train_new.py exactly)
# ============================================================================
VALUE_MAP = {
    0: 0, 100: 1, 200: 2, 300: 3, 500: 4,
    550: 5, 600: 6, 700: 7, 800: 8, 7100: 9, 10000: 10,
}
N_CLASSES = 11
CLASS_NAMES = [
    "Background", "Trees", "Lush Bushes", "Dry Grass", "Dry Bushes",
    "Ground Clutter", "Flowers", "Logs", "Rocks", "Landscape", "Sky",
]

COLOR_PALETTE = np.array([
    [0, 0, 0],        # Background - black
    [34, 139, 34],    # Trees - forest green
    [0, 200, 0],      # Lush Bushes - green
    [210, 180, 140],  # Dry Grass - tan
    [139, 90, 43],    # Dry Bushes - brown
    [128, 128, 0],    # Ground Clutter - olive
    [255, 165, 0],    # Flowers - orange
    [139, 69, 19],    # Logs - saddle brown
    [128, 128, 128],  # Rocks - gray
    [160, 82, 45],    # Landscape - sienna
    [135, 206, 235],  # Sky - sky blue
], dtype=np.uint8)


def convert_mask(mask_img: Image.Image) -> np.ndarray:
    arr = np.array(mask_img, dtype=np.int32)
    out = np.zeros_like(arr, dtype=np.uint8)
    for raw, cls in VALUE_MAP.items():
        out[arr == raw] = cls
    return out


def mask_to_color(mask_np: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES):
        rgb[mask_np == c] = COLOR_PALETTE[c]
    return rgb


# ============================================================================
# DATASET (test-time: no augmentation, just resize + normalize)
# ============================================================================
class TestDataset(Dataset):
    def __init__(self, data_dir: str, img_size: int):
        self.img_dir = os.path.join(data_dir, "Color_Images")
        self.msk_dir = os.path.join(data_dir, "Segmentation")
        self.img_size = img_size
        self.ids = sorted(f for f in os.listdir(self.img_dir)
                          if f.lower().endswith((".png", ".jpg", ".jpeg")))
        assert len(self.ids) > 0, f"No images found in {self.img_dir}"

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        msk = Image.open(os.path.join(self.msk_dir, name))
        msk_np = convert_mask(msk)

        # Resize
        img = TF.resize(img, (self.img_size, self.img_size),
                         interpolation=Image.BILINEAR)
        msk_pil = Image.fromarray(msk_np)
        msk_pil = TF.resize(msk_pil, (self.img_size, self.img_size),
                             interpolation=Image.NEAREST)

        # To tensor + normalize
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, [0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225])
        msk_t = torch.from_numpy(np.array(msk_pil)).long()

        return img_t, msk_t, name


# ============================================================================
# MODEL — must match train_new.py architecture exactly
# ============================================================================
class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 384,
                 dilations: tuple = (1, 6, 12, 18, 24)):
        super().__init__()
        self.branches = nn.ModuleList()
        for d in dilations:
            ks = 1 if d == 1 else 3
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, ks, padding=d if d > 1 else 0,
                          dilation=d, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        total_in = out_ch * (len(dilations) + 1)
        self.project = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.gap(x), (h, w),
                                   mode="bilinear", align_corners=False))
        return self.project(torch.cat(feats, dim=1))


class DeepLabV3PlusHead(nn.Module):
    def __init__(self, in_ch: int, num_classes: int,
                 token_h: int, token_w: int,
                 aspp_ch: int = 384):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w

        self.input_proj = nn.Sequential(
            nn.Conv2d(in_ch, aspp_ch, 1, bias=False),
            nn.BatchNorm2d(aspp_ch), nn.ReLU(inplace=True),
        )
        self.aspp = ASPP(aspp_ch, aspp_ch)

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(aspp_ch, aspp_ch // 16),
            nn.ReLU(inplace=True),
            nn.Linear(aspp_ch // 16, aspp_ch),
            nn.Sigmoid(),
        )

        self.dec1 = nn.Sequential(
            nn.Conv2d(aspp_ch, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(128, num_classes, 1)
        self.shortcut = nn.Conv2d(aspp_ch, 256, 1, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.token_h, self.token_w, C).permute(0, 3, 1, 2)
        x = self.input_proj(x)
        x = self.aspp(x)

        se = self.se(x).view(x.size(0), x.size(1), 1, 1)
        x = x * se

        res = self.shortcut(x)
        x = self.dec1(x)
        x = self.dec2(x) + res
        x = self.dec3(x)
        return self.classifier(x)


# ============================================================================
# METRICS
# ============================================================================
def iou_per_class(pred_logits, target, num_classes=N_CLASSES):
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt = target.view(-1)
    ious = []
    for c in range(num_classes):
        p = pred == c
        t = tgt == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append((inter / union).item() if union > 0 else float("nan"))
    return ious


def dice_per_class(pred_logits, target, num_classes=N_CLASSES, smooth=1e-6):
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt = target.view(-1)
    dices = []
    for c in range(num_classes):
        p = pred == c
        t = tgt == c
        inter = (p & t).sum().float()
        score = (2.0 * inter + smooth) / (p.sum().float() + t.sum().float() + smooth)
        dices.append(score.item())
    return dices


def pixel_acc(pred_logits, target):
    return (torch.argmax(pred_logits, 1) == target).float().mean().item()


def safe_mean(values):
    v = [x for x in values if not math.isnan(x)]
    return float(np.mean(v)) if v else 0.0


# ============================================================================
# TEST-TIME AUGMENTATION (TTA)
# ============================================================================
@torch.no_grad()
def tta_predict(backbone, head, imgs, size):
    """Average predictions over original + horizontal flip."""
    feats = backbone.forward_features(imgs)["x_norm_patchtokens"]
    logits = head(feats)
    logits = F.interpolate(logits, size, mode="bilinear", align_corners=False)

    imgs_f = torch.flip(imgs, dims=[3])
    feats_f = backbone.forward_features(imgs_f)["x_norm_patchtokens"]
    logits_f = head(feats_f)
    logits_f = F.interpolate(logits_f, size, mode="bilinear", align_corners=False)
    logits_f = torch.flip(logits_f, dims=[3])

    return (F.softmax(logits, 1) + F.softmax(logits_f, 1)) / 2.0


# ============================================================================
# VISUALIZATION
# ============================================================================
def save_prediction_comparison(img_tensor, gt_mask, pred_mask, output_path, data_id):
    """Save side-by-side: input | ground truth | prediction."""
    img = img_tensor.cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.moveaxis(img, 0, -1)
    img = img * std + mean
    img = np.clip(img, 0, 1)

    gt_color = mask_to_color(gt_mask.cpu().numpy().astype(np.uint8))
    pred_color = mask_to_color(pred_mask.cpu().numpy().astype(np.uint8))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img);          axes[0].set_title("Input Image");    axes[0].axis("off")
    axes[1].imshow(gt_color);     axes[1].set_title("Ground Truth");   axes[1].axis("off")
    axes[2].imshow(pred_color);   axes[2].set_title("Prediction");     axes[2].axis("off")

    plt.suptitle(f"Sample: {data_id}", fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def save_metrics_summary(results, output_dir):
    """Save metrics text file + per-class bar charts."""
    os.makedirs(output_dir, exist_ok=True)

    filepath = os.path.join(output_dir, "evaluation_metrics.txt")
    with open(filepath, "w") as f:
        f.write("EVALUATION RESULTS — v3 Test\n")
        f.write("=" * 55 + "\n")
        f.write(f"Mean IoU:          {results['mean_iou']:.4f}\n")
        f.write(f"Mean Dice:         {results['mean_dice']:.4f}\n")
        f.write(f"Pixel Accuracy:    {results['mean_pixel_acc']:.4f}\n")
        f.write("=" * 55 + "\n\n")

        f.write("Per-Class IoU:\n")
        f.write("-" * 45 + "\n")
        for name, iou in zip(CLASS_NAMES, results["class_iou"]):
            iou_str = f"{iou:.4f}" if not math.isnan(iou) else "N/A"
            bar = "█" * int(iou * 25) if not math.isnan(iou) else ""
            f.write(f"  {name:<20}: {iou_str}  {bar}\n")

        f.write(f"\n{'Per-Class Dice:'}\n")
        f.write("-" * 45 + "\n")
        for name, dice in zip(CLASS_NAMES, results["class_dice"]):
            f.write(f"  {name:<20}: {dice:.4f}\n")

    print(f"\nSaved evaluation metrics to {filepath}")

    # Per-class IoU bar chart
    fig, ax = plt.subplots(figsize=(12, 7))
    valid_iou = [v if not math.isnan(v) else 0 for v in results["class_iou"]]
    bars = ax.barh(range(N_CLASSES), valid_iou,
                   color=[COLOR_PALETTE[c] / 255 for c in range(N_CLASSES)],
                   edgecolor="black")
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(CLASS_NAMES, fontsize=10)
    ax.set_xlim(0, 1)
    ax.axvline(x=results["mean_iou"], color="red", linestyle="--",
               label=f"Mean IoU = {results['mean_iou']:.4f}")
    ax.set_title(f"Per-Class IoU  (Mean: {results['mean_iou']:.4f})")
    ax.legend(fontsize=10)
    ax.grid(axis="x", alpha=0.3)

    # Add value labels on bars
    for i, v in enumerate(valid_iou):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "per_class_metrics.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved per-class metrics chart to '{output_dir}/per_class_metrics.png'")


# ============================================================================
# MAIN
# ============================================================================
def main():
    script_dir = os.path.dirname(os.path.abspath(__file__))

    parser = argparse.ArgumentParser(
        description="Segmentation Test/Inference v3 — companion to train_new.py")
    parser.add_argument("--model_path", type=str,
                        default=os.path.join(script_dir, "seg_output_v3", "best_model_v3.pth"),
                        help="Path to v3 checkpoint (.pth)")
    parser.add_argument("--data_dir", type=str,
                        default=os.path.join(script_dir, "..", "Offroad_Segmentation_testImages"),
                        help="Path to test dataset (with Color_Images/ and Segmentation/ subdirs)")
    parser.add_argument("--output_dir", type=str, default="./predictions_v3",
                        help="Directory to save outputs")
    parser.add_argument("--batch_size", type=int, default=4,
                        help="Batch size for inference")
    parser.add_argument("--num_samples", type=int, default=10,
                        help="Number of side-by-side comparison images to save")
    parser.add_argument("--use_tta", action="store_true", default=True,
                        help="Enable test-time augmentation (H-flip)")
    parser.add_argument("--no_tta", action="store_true",
                        help="Disable test-time augmentation")
    parser.add_argument("--num_workers", type=int, default=2)
    args = parser.parse_args()

    if args.no_tta:
        args.use_tta = False

    # ── device ────────────────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"Using device: {device}")

    # ── load checkpoint & extract config ──────────────────────────────────────
    print(f"Loading checkpoint from {args.model_path} ...")
    checkpoint = torch.load(args.model_path, map_location=device, weights_only=False)

    ckpt_cfg = checkpoint.get("config", {})
    backbone_size = ckpt_cfg.get("backbone", "large")
    img_size      = ckpt_cfg.get("img_size", 518)
    aspp_channels = ckpt_cfg.get("aspp_channels", 384)
    embed_dim     = ckpt_cfg.get("embed_dim", 1024)
    token_h       = ckpt_cfg.get("token_h", img_size // 14)
    token_w       = ckpt_cfg.get("token_w", img_size // 14)
    n_classes     = ckpt_cfg.get("n_classes", N_CLASSES)
    ckpt_epoch    = checkpoint.get("epoch", "?")
    ckpt_val_iou  = checkpoint.get("val_iou", "?")

    print(f"  Checkpoint epoch:    {ckpt_epoch}")
    print(f"  Checkpoint val IoU:  {ckpt_val_iou}")
    print(f"  Backbone:            DINOv2-{backbone_size.capitalize()}")
    print(f"  Input resolution:    {img_size}x{img_size}")
    print(f"  ASPP channels:       {aspp_channels}")
    print(f"  Token grid:          {token_h}x{token_w}")
    print(f"  Num classes:         {n_classes}")
    print(f"  TTA enabled:         {args.use_tta}")

    # ── dataset ───────────────────────────────────────────────────────────────
    print(f"\nLoading dataset from {args.data_dir} ...")
    test_set = TestDataset(args.data_dir, img_size)
    test_loader = DataLoader(
        test_set, batch_size=args.batch_size, shuffle=False,
        num_workers=args.num_workers, pin_memory=True,
    )
    print(f"Loaded {len(test_set)} test samples")

    # ── backbone ──────────────────────────────────────────────────────────────
    backbone_names = {
        "small": "dinov2_vits14",
        "base":  "dinov2_vitb14_reg",
        "large": "dinov2_vitl14_reg",
    }
    print(f"\nLoading DINOv2 backbone ({backbone_names[backbone_size]}) ...")
    backbone = torch.hub.load("facebookresearch/dinov2",
                              backbone_names[backbone_size],
                              pretrained=True)
    backbone.to(device)
    backbone.eval()
    print("Backbone loaded successfully!")

    # ── segmentation head ─────────────────────────────────────────────────────
    head = DeepLabV3PlusHead(
        embed_dim, n_classes, token_h, token_w,
        aspp_ch=aspp_channels,
    )
    head.load_state_dict(checkpoint["head_state_dict"])
    head.to(device)
    head.eval()
    print("Segmentation head loaded successfully!")

    # ── output directories ────────────────────────────────────────────────────
    os.makedirs(args.output_dir, exist_ok=True)
    masks_dir       = os.path.join(args.output_dir, "masks")
    masks_color_dir = os.path.join(args.output_dir, "masks_color")
    comparisons_dir = os.path.join(args.output_dir, "comparisons")
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(masks_color_dir, exist_ok=True)
    os.makedirs(comparisons_dir, exist_ok=True)

    # ── inference ─────────────────────────────────────────────────────────────
    print(f"\nRunning inference on {len(test_set)} images ...")
    size = (img_size, img_size)

    all_iou   = [[] for _ in range(n_classes)]
    all_dice  = [[] for _ in range(n_classes)]
    all_acc   = []
    sample_count = 0

    with torch.no_grad():
        pbar = tqdm(test_loader, desc="Processing", unit="batch")
        for imgs, labels, data_ids in pbar:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                if args.use_tta:
                    probs = tta_predict(backbone, head, imgs, size)
                    outputs = probs
                else:
                    feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                    logits = head(feats)
                    outputs = F.interpolate(logits, size,
                                            mode="bilinear", align_corners=False)

            predicted_masks = torch.argmax(outputs, dim=1)

            # Metrics
            batch_iou  = iou_per_class(outputs, labels, n_classes)
            batch_dice = dice_per_class(outputs, labels, n_classes)
            batch_acc  = pixel_acc(outputs, labels)

            for c in range(n_classes):
                all_iou[c].append(batch_iou[c])
                all_dice[c].append(batch_dice[c])
            all_acc.append(batch_acc)

            # Save predictions for every image
            for i in range(imgs.shape[0]):
                data_id   = data_ids[i]
                base_name = os.path.splitext(data_id)[0]

                # Raw prediction mask (class IDs 0-10)
                pred_mask = predicted_masks[i].cpu().numpy().astype(np.uint8)
                Image.fromarray(pred_mask).save(
                    os.path.join(masks_dir, f"{base_name}_pred.png"))

                # Colored prediction mask
                pred_color = mask_to_color(pred_mask)
                cv2.imwrite(
                    os.path.join(masks_color_dir, f"{base_name}_pred_color.png"),
                    cv2.cvtColor(pred_color, cv2.COLOR_RGB2BGR))

                # Side-by-side comparison (first N samples)
                if sample_count < args.num_samples:
                    save_prediction_comparison(
                        imgs[i], labels[i], predicted_masks[i],
                        os.path.join(comparisons_dir,
                                     f"sample_{sample_count:03d}_{base_name}.png"),
                        data_id,
                    )
                sample_count += 1

            pbar.set_postfix(iou=f"{safe_mean(batch_iou):.3f}",
                             acc=f"{batch_acc:.3f}")

    # ── aggregate results ─────────────────────────────────────────────────────
    class_iou  = [safe_mean(all_iou[c]) for c in range(n_classes)]
    class_dice = [safe_mean(all_dice[c]) for c in range(n_classes)]
    mean_iou_val  = safe_mean(class_iou)
    mean_dice_val = safe_mean(class_dice)
    mean_acc_val  = float(np.mean(all_acc))

    results = {
        "mean_iou":       mean_iou_val,
        "mean_dice":      mean_dice_val,
        "mean_pixel_acc": mean_acc_val,
        "class_iou":      class_iou,
        "class_dice":     class_dice,
    }

    # ── print results ─────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("EVALUATION RESULTS")
    print("=" * 55)
    print(f"  Mean IoU:          {mean_iou_val:.4f}")
    print(f"  Mean Dice:         {mean_dice_val:.4f}")
    print(f"  Pixel Accuracy:    {mean_acc_val:.4f}")
    print("-" * 55)
    print("  Per-Class IoU:")
    for name, iou, dice in zip(CLASS_NAMES, class_iou, class_dice):
        iou_str = f"{iou:.4f}" if not math.isnan(iou) else "N/A  "
        bar = "█" * int(iou * 25) if not math.isnan(iou) else ""
        print(f"    {name:<20} IoU={iou_str}  Dice={dice:.4f}  {bar}")
    print("=" * 55)

    # ── save results ──────────────────────────────────────────────────────────
    save_metrics_summary(results, args.output_dir)

    print(f"\nInference complete! Processed {len(test_set)} images.")
    print(f"\nOutputs saved to {args.output_dir}/")
    print(f"  - masks/           : Raw prediction masks (class IDs 0-{n_classes-1})")
    print(f"  - masks_color/     : Colored prediction masks (RGB)")
    print(f"  - comparisons/     : Side-by-side comparisons ({min(args.num_samples, len(test_set))} samples)")
    print(f"  - evaluation_metrics.txt")
    print(f"  - per_class_metrics.png")


if __name__ == "__main__":
    main()


Writing test.py


In [19]:
!python test.py \
--model_path ./seg_output_v3/best_model_v3.pth \
--data_dir ./Offroad_Segmentation_testImages \
--output_dir ./predictions_v3

Using device: cuda
Loading checkpoint from ./seg_output_v3/best_model_v3.pth ...
  Checkpoint epoch:    24
  Checkpoint val IoU:  0.42538576972191355
  Backbone:            DINOv2-Large
  Input resolution:    518x518
  ASPP channels:       384
  Token grid:          37x37
  Num classes:         11
  TTA enabled:         True

Loading dataset from ./Offroad_Segmentation_testImages ...
Loaded 1002 test samples

Loading DINOv2 backbone (dinov2_vitl14_reg) ...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarnin

In [20]:
!ls

Offroad_Segmentation_testImages		   seg_output_v3
Offroad_Segmentation_testImages.zip	   seg_output_v3.zip
Offroad_Segmentation_Training_Dataset	   test.py
Offroad_Segmentation_Training_Dataset.zip  train_new.py
predictions_v3


In [21]:
!zip -r predictions_v3.zip predictions_v3

  adding: predictions_v3/ (stored 0%)
  adding: predictions_v3/comparisons/ (stored 0%)
  adding: predictions_v3/comparisons/sample_002_0000062.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_007_0000067.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_000_0000060.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_005_0000065.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_004_0000064.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_009_0000069.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_003_0000063.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_006_0000066.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_008_0000068.png (deflated 0%)
  adding: predictions_v3/comparisons/sample_001_0000061.png (deflated 0%)
  adding: predictions_v3/per_class_metrics.png (deflated 23%)
  adding: predictions_v3/evaluation_metrics.txt (deflated 72%)
  adding: predictions_v3/masks_color/ (stored 0

In [22]:
from IPython.display import FileLink
FileLink('predictions_v3.zip')

/kaggle/working/predictions_v3.zip

In [23]:
!ls

Offroad_Segmentation_testImages		   predictions_v3      test.py
Offroad_Segmentation_testImages.zip	   predictions_v3.zip  train_new.py
Offroad_Segmentation_Training_Dataset	   seg_output_v3
Offroad_Segmentation_Training_Dataset.zip  seg_output_v3.zip


In [24]:
%%writefile test.py
"""
Segmentation Test / Inference Script v3
========================================
Companion to train_new.py (v3 Colab MAX).
Loads a v3 checkpoint (DeepLabV3+ head on DINOv2 backbone) and runs
evaluation + prediction saving on a test dataset.

Supports:
  - Auto-config from checkpoint (backbone, img_size, aspp_channels, etc.)
  - Test-Time Augmentation (TTA): H-flip ensemble
  - Per-class IoU, Dice, Pixel Accuracy
  - Saves: raw masks, colored masks, side-by-side comparisons, metrics

USAGE:
  python test_new.py \
      --model_path ./seg_output_v3/best_model_v3.pth \
      --data_dir   ../Offroad_Segmentation_testImages \
      --output_dir ./predictions_v3
"""

import os, math, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm


# ============================================================================
# CLASS DEFINITIONS  (must match train_new.py exactly)
# ============================================================================
VALUE_MAP = {
    0: 0, 100: 1, 200: 2, 300: 3, 500: 4,
    550: 5, 600: 6, 700: 7, 800: 8, 7100: 9, 10000: 10,
}
N_CLASSES = 11
CLASS_NAMES = [
    "Background", "Trees", "Lush Bushes", "Dry Grass", "Dry Bushes",
    "Ground Clutter", "Flowers", "Logs", "Rocks", "Landscape", "Sky",
]

COLOR_PALETTE = np.array([
    [0, 0, 0],        # Background - black
    [34, 139, 34],    # Trees - forest green
    [0, 200, 0],      # Lush Bushes - green
    [210, 180, 140],  # Dry Grass - tan
    [139, 90, 43],    # Dry Bushes - brown
    [128, 128, 0],    # Ground Clutter - olive
    [255, 165, 0],    # Flowers - orange
    [139, 69, 19],    # Logs - saddle brown
    [128, 128, 128],  # Rocks - gray
    [160, 82, 45],    # Landscape - sienna
    [135, 206, 235],  # Sky - sky blue
], dtype=np.uint8)


def convert_mask(mask_img: Image.Image) -> np.ndarray:
    arr = np.array(mask_img, dtype=np.int32)
    out = np.zeros_like(arr, dtype=np.uint8)
    for raw, cls in VALUE_MAP.items():
        out[arr == raw] = cls
    return out


def mask_to_color(mask_np: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES):
        rgb[mask_np == c] = COLOR_PALETTE[c]
    return rgb


# ============================================================================
# DATASET (test-time: no augmentation, just resize + normalize)
# ============================================================================
class TestDataset(Dataset):
    def __init__(self, data_dir: str, img_size: int):
        self.img_dir = os.path.join(data_dir, "Color_Images")
        self.msk_dir = os.path.join(data_dir, "Segmentation")
        self.img_size = img_size
        self.ids = sorted(f for f in os.listdir(self.img_dir)
                          if f.lower().endswith((".png", ".jpg", ".jpeg")))
        assert len(self.ids) > 0, f"No images found in {self.img_dir}"

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        msk = Image.open(os.path.join(self.msk_dir, name))
        msk_np = convert_mask(msk)

        # Resize
        img = TF.resize(img, (self.img_size, self.img_size),
                         interpolation=Image.BILINEAR)
        msk_pil = Image.fromarray(msk_np)
        msk_pil = TF.resize(msk_pil, (self.img_size, self.img_size),
                             interpolation=Image.NEAREST)

        # To tensor + normalize
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, [0.485, 0.456, 0.406],
                              [0.229, 0.224, 0.225])
        msk_t = torch.from_numpy(np.array(msk_pil)).long()

        return img_t, msk_t, name


# ============================================================================
# MODEL — must match train_new.py architecture exactly
# ============================================================================
class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 384,
                 dilations: tuple = (1, 6, 12, 18, 24)):
        super().__init__()
        self.branches = nn.ModuleList()
        for d in dilations:
            ks = 1 if d == 1 else 3
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, ks, padding=d if d > 1 else 0,
                          dilation=d, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        total_in = out_ch * (len(dilations) + 1)
        self.project = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.gap(x), (h, w),
                                   mode="bilinear", align_corners=False))
        return self.project(torch.cat(feats, dim=1))


class DeepLabV3PlusHead(nn.Module):
    def __init__(self, in_ch: int, num_classes: int,
                 token_h: int, token_w: int,
                 aspp_ch: int = 384):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w

        self.input_proj = nn.Sequential(
            nn.Conv2d(in_ch, aspp_ch, 1, bias=False),
            nn.BatchNorm2d(aspp_ch), nn.ReLU(inplace=True),
        )
        self.aspp = ASPP(aspp_ch, aspp_ch)

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(aspp_ch, aspp_ch // 16),
            nn.ReLU(inplace=True),
            nn.Linear(aspp_ch // 16, aspp_ch),
            nn.Sigmoid(),
        )

        self.dec1 = nn.Sequential(
            nn.Conv2d(aspp_ch, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(128, num_classes, 1)
        self.shortcut = nn.Conv2d(aspp_ch, 256, 1, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.token_h, self.token_w, C).permute(0, 3, 1, 2)
        x = self.input_proj(x)
        x = self.aspp(x)

        se = self.se(x).view(x.size(0), x.size(1), 1, 1)
        x = x * se

        res = self.shortcut(x)
        x = self.dec1(x)
        x = self.dec2(x) + res
        x = self.dec3(x)
        return self.classifier(x)


# ============================================================================
# METRICS
# ============================================================================
def iou_per_class(pred_logits, target, num_classes=N_CLASSES):
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt = target.view(-1)
    ious = []
    for c in range(num_classes):
        p = pred == c
        t = tgt == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append((inter / union).item() if union > 0 else float("nan"))
    return ious


def dice_per_class(pred_logits, target, num_classes=N_CLASSES, smooth=1e-6):
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt = target.view(-1)
    dices = []
    for c in range(num_classes):
        p = pred == c
        t = tgt == c
        inter = (p & t).sum().float()
        score = (2.0 * inter + smooth) / (p.sum().float() + t.sum().float() + smooth)
        dices.append(score.item())
    return dices


def pixel_acc(pred_logits, target):
    return (torch.argmax(pred_logits, 1) == target).float().mean().item()


def safe_mean(values):
    v = [x for x in values if not math.isnan(x)]
    return float(np.mean(v)) if v else 0.0


def clean_state_dict(state_dict):
    """Remove common training wrappers from checkpoint keys."""
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith("module."):
            key = key[len("module."):]
        if key.startswith("_orig_mod."):
            key = key[len("_orig_mod."):]
        cleaned[key] = value
    return cleaned


def extract_head_state_dict(checkpoint):
    """Support both full checkpoints and plain uploaded state_dict files."""
    if isinstance(checkpoint, dict):
        if "head_state_dict" in checkpoint:
            return clean_state_dict(checkpoint["head_state_dict"]), checkpoint.get("config", {})

        for candidate_key in ("state_dict", "model_state_dict", "model", "net"):
            candidate = checkpoint.get(candidate_key)
            if isinstance(candidate, dict):
                tensor_values = [v for v in candidate.values() if torch.is_tensor(v)]
                if tensor_values:
                    return clean_state_dict(candidate), checkpoint.get("config", {})

        tensor_values = [v for v in checkpoint.values() if torch.is_tensor(v)]
        if tensor_values:
            return clean_state_dict(checkpoint), checkpoint.get("config", {})

    raise ValueError("Unsupported checkpoint format. Expected a full checkpoint or a plain state_dict.")


# ============================================================================
# TEST-TIME AUGMENTATION (TTA)
# ============================================================================
@torch.no_grad()
def tta_predict(backbone, head, imgs, size):
    """Average predictions over original + horizontal flip."""
    feats = backbone.forward_features(imgs)["x_norm_patchtokens"]
    logits = head(feats)
    logits = F.interpolate(logits, size, mode="bilinear", align_corners=False)

    imgs_f = torch.flip(imgs, dims=[3])
    feats_f = backbone.forward_features(imgs_f)["x_norm_patchtokens"]
    logits_f = head(feats_f)
    logits_f = F.interpolate(logits_f, size, mode="bilinear", align_corners=False)
    logits_f = torch.flip(logits_f, dims=[3])

    return (F.softmax(logits, 1) + F.softmax(logits_f, 1)) / 2.0


# ============================================================================
# VISUALIZATION
# ============================================================================
def save_prediction_comparison(img_tensor, gt_mask, pred_mask, output_path, data_id):
    """Save side-by-side: input | ground truth | prediction."""
    img = img_tensor.cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.moveaxis(img, 0, -1)
    img = img * std + mean
    img = np.clip(img, 0, 1)

    gt_color = mask_to_color(gt_mask.cpu().numpy().astype(np.uint8))
    pred_color = mask_to_color(pred_mask.cpu().numpy().astype(np.uint8))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img);          axes[0].set_title("Input Image");    axes[0].axis("off")
    axes[1].imshow(gt_color);     axes[1].set_title("Ground Truth");   axes[1].axis("off")
    axes[2].imshow(pred_color);   axes[2].set_title("Prediction");     axes[2].axis("off")

    plt.suptitle(f"Sample: {data_id}", fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def save_metrics_summary(results, output_dir):
    """Save metrics text file + per-class bar charts."""
    os.makedirs(output_dir, exist_ok=True)

    filepath = os.path.join(output_dir, "evaluation_metrics.txt")
    with open(filepath, "w") as f:
        f.write("EVALUATION RESULTS — v3 Test\n")
        f.write("=" * 55 + "\n")
        f.write(f"Mean IoU:          {results['mean_iou']:.4f}\n")
        f.write(f"Mean Dice:         {results['mean_dice']:.4f}\n")
        f.write(f"Pixel Accuracy:    {results['mean_pixel_acc']:.4f}\n")
        f.write("=" * 55 + "\n\n")

        f.write("Per-Class IoU:\n")
        f.write("-" * 45 + "\n")
        for name, iou in zip(CLASS_NAMES, results["class_iou"]):
            iou_str = f"{iou:.4f}" if not math.isnan(iou) else "N/A"
            bar = "█" * int(iou * 25) if not math.isnan(iou) else ""
            f.write(f"  {name:<20}: {iou_str}  {bar}\n")

        f.write(f"\n{'Per-Class Dice:'}\n")
        f.write("-" * 45 + "\n")
        for name, dice in zip(CLASS_NAMES, results["class_dice"]):
            f.write(f"  {name:<20}: {dice:.4f}\n")

    print(f"\nSaved evaluation metrics to {filepath}")

    # Per-class IoU bar chart
    fig, ax = plt.subplots(figsize=(12, 7))
    valid_iou = [v if not math.isnan(v) else 0 for v in results["class_iou"]]
    bars = ax.barh(range(N_CLASSES), valid_iou,
                   color=[COLOR_PALETTE[c] / 255 for c in range(N_CLASSES)],
                   edgecolor="black")
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(CLASS_NAMES, fontsize=10)
    ax.set_xlim(0, 1)
    ax.axvline(x=results["mean_iou"], color="red", linestyle="--",
               label=f"Mean IoU = {results['mean_iou']:.4f}")
    ax.set_title(f"Per-Class IoU  (Mean: {results['mean_iou']:.4f})")
    ax.legend(fontsize=10)
    ax.grid(axis="x", alpha=0.3)

    # Add value labels on bars
    for i, v in enumerate(valid_iou):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "per_class_metrics.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved per-class metrics chart to '{output_dir}/per_class_metrics.png'")


# ============================================================================
# MAIN
# ============================================================================
def main():
    script_dir = os.path.dirname(os.path.abspath(__file__))
    kaggle_model_path = "/kaggle/input/models/purohitdarshan/hao/pytorch/default/1/model_final3.pth"
    kaggle_data_dir = "/kaggle/working/Offroad_Segmentation_testImages"
    kaggle_output_dir = "/kaggle/working/predictions_v3"

    parser = argparse.ArgumentParser(
        description="Segmentation Test/Inference v3 — companion to train_new.py")
    parser.add_argument("--model_path", type=str,
                        default=kaggle_model_path,
                        help="Path to v3 checkpoint (.pth)")
    parser.add_argument("--data_dir", type=str,
                        default=kaggle_data_dir,
                        help="Path to test dataset (with Color_Images/ and Segmentation/ subdirs)")
    parser.add_argument("--output_dir", type=str, default=kaggle_output_dir,
                        help="Directory to save outputs")
    parser.add_argument("--batch_size", type=int, default=4,
                        help="Batch size for inference")
    parser.add_argument("--num_samples", type=int, default=10,
                        help="Number of side-by-side comparison images to save")
    parser.add_argument("--use_tta", action="store_true", default=True,
                        help="Enable test-time augmentation (H-flip)")
    parser.add_argument("--no_tta", action="store_true",
                        help="Disable test-time augmentation")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--backbone", type=str, default="large",
                        choices=["small", "base", "large"],
                        help="Fallback backbone if checkpoint config is missing")
    parser.add_argument("--img_size", type=int, default=518,
                        help="Fallback image size if checkpoint config is missing")
    parser.add_argument("--aspp_channels", type=int, default=384,
                        help="Fallback ASPP channels if checkpoint config is missing")
    parser.add_argument("--embed_dim", type=int, default=1024,
                        help="Fallback embedding dimension if checkpoint config is missing")
    parser.add_argument("--n_classes", type=int, default=N_CLASSES,
                        help="Fallback class count if checkpoint config is missing")
    args = parser.parse_args()

    if args.no_tta:
        args.use_tta = False

    # ── device ────────────────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"Using device: {device}")

    # ── load checkpoint & extract config ──────────────────────────────────────
    print(f"Loading checkpoint from {args.model_path} ...")
    checkpoint = torch.load(args.model_path, map_location=device, weights_only=False)

    head_state_dict, ckpt_cfg = extract_head_state_dict(checkpoint)
    backbone_size = ckpt_cfg.get("backbone", args.backbone)
    img_size      = ckpt_cfg.get("img_size", args.img_size)
    aspp_channels = ckpt_cfg.get("aspp_channels", args.aspp_channels)
    embed_dim     = ckpt_cfg.get("embed_dim", args.embed_dim)
    token_h       = ckpt_cfg.get("token_h", img_size // 14)
    token_w       = ckpt_cfg.get("token_w", img_size // 14)
    n_classes     = ckpt_cfg.get("n_classes", args.n_classes)
    ckpt_epoch    = checkpoint.get("epoch", "?") if isinstance(checkpoint, dict) else "?"
    ckpt_val_iou  = checkpoint.get("val_iou", "?") if isinstance(checkpoint, dict) else "?"
    has_embedded_config = bool(ckpt_cfg)

    print(f"  Checkpoint epoch:    {ckpt_epoch}")
    print(f"  Checkpoint val IoU:  {ckpt_val_iou}")
    print(f"  Embedded config:     {has_embedded_config}")
    print(f"  Backbone:            DINOv2-{backbone_size.capitalize()}")
    print(f"  Input resolution:    {img_size}x{img_size}")
    print(f"  ASPP channels:       {aspp_channels}")
    print(f"  Token grid:          {token_h}x{token_w}")
    print(f"  Num classes:         {n_classes}")
    print(f"  TTA enabled:         {args.use_tta}")

    # ── dataset ───────────────────────────────────────────────────────────────
    print(f"\nLoading dataset from {args.data_dir} ...")
    test_set = TestDataset(args.data_dir, img_size)
    test_loader = DataLoader(
        test_set, batch_size=args.batch_size, shuffle=False,
        num_workers=args.num_workers, pin_memory=True,
    )
    print(f"Loaded {len(test_set)} test samples")

    # ── backbone ──────────────────────────────────────────────────────────────
    backbone_names = {
        "small": "dinov2_vits14",
        "base":  "dinov2_vitb14_reg",
        "large": "dinov2_vitl14_reg",
    }
    print(f"\nLoading DINOv2 backbone ({backbone_names[backbone_size]}) ...")
    backbone = torch.hub.load("facebookresearch/dinov2",
                              backbone_names[backbone_size],
                              pretrained=True)
    backbone.to(device)
    backbone.eval()
    print("Backbone loaded successfully!")

    # ── segmentation head ─────────────────────────────────────────────────────
    head = DeepLabV3PlusHead(
        embed_dim, n_classes, token_h, token_w,
        aspp_ch=aspp_channels,
    )
    head.load_state_dict(head_state_dict)
    head.to(device)
    head.eval()
    print("Segmentation head loaded successfully!")

    # ── output directories ────────────────────────────────────────────────────
    os.makedirs(args.output_dir, exist_ok=True)
    masks_dir       = os.path.join(args.output_dir, "masks")
    masks_color_dir = os.path.join(args.output_dir, "masks_color")
    comparisons_dir = os.path.join(args.output_dir, "comparisons")
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(masks_color_dir, exist_ok=True)
    os.makedirs(comparisons_dir, exist_ok=True)

    # ── inference ─────────────────────────────────────────────────────────────
    print(f"\nRunning inference on {len(test_set)} images ...")
    size = (img_size, img_size)

    all_iou   = [[] for _ in range(n_classes)]
    all_dice  = [[] for _ in range(n_classes)]
    all_acc   = []
    sample_count = 0

    with torch.no_grad():
        pbar = tqdm(test_loader, desc="Processing", unit="batch")
        for imgs, labels, data_ids in pbar:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                if args.use_tta:
                    probs = tta_predict(backbone, head, imgs, size)
                    outputs = probs
                else:
                    feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                    logits = head(feats)
                    outputs = F.interpolate(logits, size,
                                            mode="bilinear", align_corners=False)

            predicted_masks = torch.argmax(outputs, dim=1)

            # Metrics
            batch_iou  = iou_per_class(outputs, labels, n_classes)
            batch_dice = dice_per_class(outputs, labels, n_classes)
            batch_acc  = pixel_acc(outputs, labels)

            for c in range(n_classes):
                all_iou[c].append(batch_iou[c])
                all_dice[c].append(batch_dice[c])
            all_acc.append(batch_acc)

            # Save predictions for every image
            for i in range(imgs.shape[0]):
                data_id   = data_ids[i]
                base_name = os.path.splitext(data_id)[0]

                # Raw prediction mask (class IDs 0-10)
                pred_mask = predicted_masks[i].cpu().numpy().astype(np.uint8)
                Image.fromarray(pred_mask).save(
                    os.path.join(masks_dir, f"{base_name}_pred.png"))

                # Colored prediction mask
                pred_color = mask_to_color(pred_mask)
                cv2.imwrite(
                    os.path.join(masks_color_dir, f"{base_name}_pred_color.png"),
                    cv2.cvtColor(pred_color, cv2.COLOR_RGB2BGR))

                # Side-by-side comparison (first N samples)
                if sample_count < args.num_samples:
                    save_prediction_comparison(
                        imgs[i], labels[i], predicted_masks[i],
                        os.path.join(comparisons_dir,
                                     f"sample_{sample_count:03d}_{base_name}.png"),
                        data_id,
                    )
                sample_count += 1

            pbar.set_postfix(iou=f"{safe_mean(batch_iou):.3f}",
                             acc=f"{batch_acc:.3f}")

    # ── aggregate results ─────────────────────────────────────────────────────
    class_iou  = [safe_mean(all_iou[c]) for c in range(n_classes)]
    class_dice = [safe_mean(all_dice[c]) for c in range(n_classes)]
    mean_iou_val  = safe_mean(class_iou)
    mean_dice_val = safe_mean(class_dice)
    mean_acc_val  = float(np.mean(all_acc))

    results = {
        "mean_iou":       mean_iou_val,
        "mean_dice":      mean_dice_val,
        "mean_pixel_acc": mean_acc_val,
        "class_iou":      class_iou,
        "class_dice":     class_dice,
    }

    # ── print results ─────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("EVALUATION RESULTS")
    print("=" * 55)
    print(f"  Mean IoU:          {mean_iou_val:.4f}")
    print(f"  Mean Dice:         {mean_dice_val:.4f}")
    print(f"  Pixel Accuracy:    {mean_acc_val:.4f}")
    print("-" * 55)
    print("  Per-Class IoU:")
    for name, iou, dice in zip(CLASS_NAMES, class_iou, class_dice):
        iou_str = f"{iou:.4f}" if not math.isnan(iou) else "N/A  "
        bar = "█" * int(iou * 25) if not math.isnan(iou) else ""
        print(f"    {name:<20} IoU={iou_str}  Dice={dice:.4f}  {bar}")
    print("=" * 55)

    # ── save results ──────────────────────────────────────────────────────────
    save_metrics_summary(results, args.output_dir)

    print(f"\nInference complete! Processed {len(test_set)} images.")
    print(f"\nOutputs saved to {args.output_dir}/")
    print(f"  - masks/           : Raw prediction masks (class IDs 0-{n_classes-1})")
    print(f"  - masks_color/     : Colored prediction masks (RGB)")
    print(f"  - comparisons/     : Side-by-side comparisons ({min(args.num_samples, len(test_set))} samples)")
    print(f"  - evaluation_metrics.txt")
    print(f"  - per_class_metrics.png")


if __name__ == "__main__":
    main()


Overwriting test.py


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import cv2
import os
from tqdm import tqdm

# ================================
# PATHS
# ================================

MODEL_PATH = "/kaggle/input/models/purohitdarshan/hao/pytorch/default/1/model_final3.pth"

IMAGE_DIR = "/kaggle/working/Offroad_Segmentation_testImages/Color_Images"

GT_DIR = "/kaggle/working/Offroad_Segmentation_testImages/Segmentation"

OUTPUT_DIR = "/kaggle/working/predictions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================
# SETTINGS
# ================================

H, W = 392, 672
TOKEN_H, TOKEN_W = 28, 48
NUM_CLASSES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ================================
# COLOR MAP
# ================================

COLOR_PALETTE = np.array([
[0,0,0],
[34,139,34],
[0,255,0],
[210,180,140],
[139,90,43],
[128,128,0],
[139,69,19],
[128,128,128],
[160,82,45],
[135,206,235],
], dtype=np.uint8)

# ================================
# MODEL ARCHITECTURE
# ================================

class DINOv2MultiScaleWrapper(nn.Module):
    def __init__(self, backbone, n_last_layers=4):
        super().__init__()
        self.backbone = backbone
        self.n_last_layers = n_last_layers

    def forward(self, x):
        features = self.backbone.get_intermediate_layers(
            x, n=self.n_last_layers, reshape=False
        )
        return list(features)


class Mask2FormerHead(nn.Module):
    def __init__(self, in_channels, num_classes, tokenH, tokenW):
        super().__init__()

        self.tokenH = tokenH
        self.tokenW = tokenW

        self.conv = nn.Conv2d(in_channels, num_classes, 1)

    def forward(self, multi_scale_features):

        feat = multi_scale_features[0]

        B,N,C = feat.shape

        feat = feat.reshape(B,self.tokenH,self.tokenW,C).permute(0,3,1,2)

        logits = self.conv(feat)

        return {"pred_logits": logits, "pred_masks": logits}


def mask2former_inference(outputs, img_size):

    masks = outputs["pred_masks"]

    masks = F.interpolate(
        masks,
        size=img_size,
        mode="bilinear",
        align_corners=False
    )

    return masks


# ================================
# LOAD BACKBONE
# ================================

print("Loading DINOv2 backbone...")

raw_backbone = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

raw_backbone.eval().to(device)

backbone = DINOv2MultiScaleWrapper(raw_backbone).to(device)

# ================================
# DETECT EMBEDDING SIZE
# ================================

dummy = torch.randn(1,3,H,W).to(device)

with torch.no_grad():
    feats = backbone(dummy)

embed_dim = feats[0].shape[2]

print("Embedding dim:",embed_dim)

# ================================
# LOAD SEGMENTATION HEAD
# ================================

model = Mask2FormerHead(
    in_channels=embed_dim,
    num_classes=NUM_CLASSES,
    tokenH=TOKEN_H,
    tokenW=TOKEN_W
).to(device)

state_dict = torch.load(MODEL_PATH,map_location=device)

model.load_state_dict(state_dict,strict=False)

model.eval()

print("Model Loaded")

# ================================
# TRANSFORM
# ================================

transform = transforms.Compose([
    transforms.Resize((H,W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ================================
# INFERENCE
# ================================

images = sorted(os.listdir(IMAGE_DIR))

print("Total images:",len(images))

for img_name in tqdm(images):

    if not img_name.endswith(".png"):
        continue

    img_path = os.path.join(IMAGE_DIR,img_name)

    image = Image.open(img_path).convert("RGB")

    original_size = image.size

    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        feats = backbone(img_tensor)

        outputs = model(feats)

        seg = mask2former_inference(outputs,(H,W))

        pred = torch.argmax(seg,1)[0].cpu().numpy()

    pred = cv2.resize(
        pred.astype(np.uint8),
        original_size,
        interpolation=cv2.INTER_NEAREST
    )

    color_mask = COLOR_PALETTE[pred]

    save_path = os.path.join(OUTPUT_DIR,img_name)

    cv2.imwrite(save_path,color_mask)

print("Inference complete")
print("Saved to:",OUTPUT_DIR)

Device: cuda
Loading DINOv2 backbone...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 266MB/s]


Embedding dim: 384
Model Loaded
Total images: 1002


100%|██████████| 1002/1002 [01:07<00:00, 14.91it/s]

Inference complete
Saved to: /kaggle/working/predictions


In [40]:
!zip -r -q predictions.zip /kaggle/working/predictions

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import cv2
import os
from tqdm import tqdm

# ==========================
# PATHS
# ==========================

MODEL_PATH = "/kaggle/input/models/purohitdarshan/hao1-0/pytorch/default/1/potato2_segmentation_head_best.pth"

IMAGE_DIR = "/kaggle/working/Offroad_Segmentation_testImages/Color_Images"

OUTPUT_DIR = "/kaggle/working/predictions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================
# SETTINGS
# ==========================

H, W = 392, 672
TOKEN_H, TOKEN_W = 28, 48
NUM_CLASSES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ==========================
# COLOR MAP
# ==========================

COLOR_PALETTE = np.array([
[0,0,0],
[34,139,34],
[0,255,0],
[210,180,140],
[139,90,43],
[128,128,0],
[139,69,19],
[128,128,128],
[160,82,45],
[135,206,235],
], dtype=np.uint8)

# ==========================
# MODEL
# ==========================

class DINOv2MultiScaleWrapper(nn.Module):
    def __init__(self, backbone, n_last_layers=4):
        super().__init__()
        self.backbone = backbone
        self.n_last_layers = n_last_layers

    def forward(self, x):
        features = self.backbone.get_intermediate_layers(
            x,
            n=self.n_last_layers,
            reshape=False
        )
        return list(features)


class SimpleDecoder(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels,256,3,padding=1)
        self.conv2 = nn.Conv2d(256,128,3,padding=1)
        self.conv3 = nn.Conv2d(128,num_classes,1)

    def forward(self,x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)

        return x


# ==========================
# LOAD BACKBONE
# ==========================

print("Loading DINOv2...")

raw_backbone = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

raw_backbone.eval().to(device)

backbone = DINOv2MultiScaleWrapper(raw_backbone).to(device)

# ==========================
# FIND EMBEDDING SIZE
# ==========================

dummy = torch.randn(1,3,H,W).to(device)

with torch.no_grad():
    feats = backbone(dummy)

embed_dim = feats[0].shape[2]

print("Embedding dim:",embed_dim)

# ==========================
# LOAD MODEL
# ==========================

model = SimpleDecoder(embed_dim,NUM_CLASSES).to(device)

state_dict = torch.load(MODEL_PATH,map_location=device)

model.load_state_dict(state_dict,strict=False)

model.eval()

print("Potato model loaded")

# ==========================
# TRANSFORM
# ==========================

transform = transforms.Compose([
    transforms.Resize((H,W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ==========================
# INFERENCE
# ==========================

images = sorted(os.listdir(IMAGE_DIR))

print("Images found:",len(images))

for img_name in tqdm(images):

    if not img_name.endswith(".png"):
        continue

    img_path = os.path.join(IMAGE_DIR,img_name)

    image = Image.open(img_path).convert("RGB")

    original_size = image.size

    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        feats = backbone(tensor)

        feat = feats[0]

        B,N,C = feat.shape

        feat = feat.reshape(B,TOKEN_H,TOKEN_W,C).permute(0,3,1,2)

        logits = model(feat)

        logits = F.interpolate(
            logits,
            size=(H,W),
            mode="bilinear",
            align_corners=False
        )

        pred = torch.argmax(logits,1)[0].cpu().numpy()

    pred = cv2.resize(
        pred.astype(np.uint8),
        original_size,
        interpolation=cv2.INTER_NEAREST
    )

    color_mask = COLOR_PALETTE[pred]

    save_path = os.path.join(OUTPUT_DIR,img_name)

    cv2.imwrite(save_path,color_mask)

print("Inference complete")
print("Saved to:",OUTPUT_DIR)

Device: cuda
Loading DINOv2...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Embedding dim: 384
Potato model loaded
Images found: 1002


100%|██████████| 1002/1002 [01:09<00:00, 14.40it/s]

Inference complete
Saved to: /kaggle/working/predictions


In [43]:
!ls predictions


0000060.png  0000227.png  0000394.png  0000561.png  0000728.png  0000895.png
0000061.png  0000228.png  0000395.png  0000562.png  0000729.png  0000896.png
0000062.png  0000229.png  0000396.png  0000563.png  0000730.png  0000897.png
0000063.png  0000230.png  0000397.png  0000564.png  0000731.png  0000898.png
0000064.png  0000231.png  0000398.png  0000565.png  0000732.png  0000899.png
0000065.png  0000232.png  0000399.png  0000566.png  0000733.png  0000900.png
0000066.png  0000233.png  0000400.png  0000567.png  0000734.png  0000901.png
0000067.png  0000234.png  0000401.png  0000568.png  0000735.png  0000902.png
0000068.png  0000235.png  0000402.png  0000569.png  0000736.png  0000903.png
0000069.png  0000236.png  0000403.png  0000570.png  0000737.png  0000904.png
0000070.png  0000237.png  0000404.png  0000571.png  0000738.png  0000905.png
0000071.png  0000238.png  0000405.png  0000572.png  0000739.png  0000906.png
0000072.png  0000239.png  0000406.png  0000573.png  0000740.png  0000907.png